In [1]:
!pip install fake_useragent


[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
from fake_useragent import UserAgent

def set_header() -> dict[str, str]:
    return {
        "User-Agent": UserAgent().random,
        "Accept-Language": "ko-KR,ko;q=0.8,en-US;q=0.5,en;q=0.3"
    }

import time
import random

def crawling_waiting_time() -> None:
    return time.sleep(random.randint(1, 3))

import requests
from bs4 import BeautifulSoup

def check_last_page(url):
    response = requests.get(url, headers=set_header())
    response.raise_for_status()
    if response.status_code == 200:
        return response.text
        # soup = BeautifulSoup(response.text, 'html.parser')
        # page = soup.find('div', class_='product-list-paging')
        # return int(page['data-total'])
    return ''

In [ ]:
r = check_last_page('https://link.coupang.com/a/cdqh0M')
r

'<!doctype html>\n<html lang="ko">\n  <head>\n    <meta charset="utf-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />\n    <title></title>\n    <link rel="shortcut icon" href="//image9.coupangcdn.com/image/coupang/favicon/v2/favicon.ico" type="image/x-icon" />\n    <link rel="apple-touch-icon-precomposed" href="//image6.coupangcdn.com/image/coupang/favicon/v2/favicon_256x256.ico" />\n    <style type="text/css">\n    html,body {\n    padding: 0;\n    margin: 0;\n    height: 100%;\n    }\n    body {\n        width: 100%;\n        height: 100%;\n        background-image: url(\'//img1a.coupangcdn.com/image/landing_images/landing_bg_20240131.png\');\n        background-size: contain;\n        background-repeat: no-repeat;\n        font-family: "Apple SD Gothic Neo", "SF Pro Display", "Noto Sans KR", "Roboto", sans-serif;\n    }\n    .redirect-section {\n        position: fixed;\n        width: 100%;\n  

In [2]:
base_url = "https://rss.blog.naver.com/{name}.xml"
count_url = "https://blog.naver.com/NVisitorgp4Ajax.nhn?blogId={name}"
blogs = [
    ("bcf5qp11", "쇼핑"), # 고풜 (1개씩)
    ("celubdigging", "파트너스활동으로 소정 수익발생"),
    ("dalcome5", "◇궁금해◇"),
    ("dehi61", "파트너스활동으로 소정 수익발생"),
    ("fashionblog-", "방송·연예·패션"),
    ("hkh443", "추천템"),
    ("hprbel1097", "방송/패션"),
    ("hsh6566", "방송아이템"),
    ("jsodnfak", "방송정보"),
    ("phd_choi93", "패션/코디 정보"),
    ("pravas", "TV 속 상품 정보"),
    ("skywhite369", "방송v패션v맛집v제품"),
    ("lzheng", "방송그제품"),
    ("unknown0998", "정보 생활"),
    ("nemo-c", "방송제품정보"),
    # ("skyblue2564", "셀럽아이템"), # 카피0 # 이미지 링크
        # ("moji1122", "셀럽옷장"), # 카피1
        # ("sanand0206", "패션탐구생활"), # 카피2
        
    # ("yangki2346", "방송,제품"), # 댓글형 링크
    # ("agla1", "유용한 아이템"), # 댓글형 링크
    # ("khm8208", "방송아이템"), # 댓글형 링크
    # ("tazsun20", "방송아이템★"), # 댓글형 링크
    # ("mn_uni", "제품정보"), # 댓글형 링크
        
    # ("mrdaddytv", "패션상품정보"), # 저품질
    # ("icis66", "방송아이템"), # 저품질
    # ("smilek17", "방송/코디"), # 저품질
]

In [3]:
def get_visit_count(name):
    import xmltodict
    import requests
    url = f"https://blog.naver.com/NVisitorgp4Ajax.nhn?blogId={name}"
    response = requests.get(url)
    tree = xmltodict.parse(response.text)
    cmax = 0
    for stree in tree["visitorcnts"]["visitorcnt"]:
        # print(int(stree["@cnt"]))
        cmax = max(int(stree["@cnt"]), cmax)        
    return cmax

In [4]:
get_visit_count('shaneangel')

138

In [5]:
import pandas as pd
df = pd.DataFrame([x[0] for x in blogs], columns=['url'])
df

,url
0,bcf5qp11
1,celubdigging
2,dalcome5
3,dehi61
4,fashionblog-
5,hkh443
6,hprbel1097
7,hsh6566
8,jsodnfak
9,phd_choi93


In [6]:
today = pd.to_datetime('now', utc=True).floor('D').strftime('%y%m%d')
today

'250207'

In [7]:
df['count'] = df['url'].apply(lambda x : get_visit_count(x))

In [8]:
df = df.sort_values('count', ascending=False)
df = df.reset_index(drop=True)
df.to_csv('blog_data/test.csv')
df

,url,count
0,nemo-c,7449
1,hsh6566,4974
2,lzheng,3852
3,phd_choi93,3240
4,skywhite369,2557
5,celubdigging,1683
6,dalcome5,1635
7,hkh443,1462
8,hprbel1097,1179
9,pravas,883


In [9]:
def get_rss(name, folder):
    import xmltodict
    import requests
    url = base_url.format(name=name)
    response = requests.get(url)
    tree = xmltodict.parse(response.text)
    # list인 item 내 category가 folder인 목록
    # return tree
    return [{'title':x['title'], 'url':x['guid'], 'date':x['pubDate'], 'tag':x['tag'] if x['tag'] != None else ''} for x in tree['rss']['channel']['item'] if x['category'] == folder]

In [10]:
get_rss("agla1", "유용한 아이템")

[{'title': '최화정 자라 가방 진주 목걸이 선글라스 하트 찍찍이 레몬즙짜개 가글',
  'url': 'https://blog.naver.com/agla1/223751135245',
  'date': 'Fri, 07 Feb 2025 12:02:37 +0900',
  'tag': ''},
 {'title': '최화정 향수 괄사 얼굴 마사지기 석쇠 미니 비닐 투명 지갑 아로마 오일 블러쉬 블러셔',
  'url': 'https://blog.naver.com/agla1/223751060216',
  'date': 'Fri, 07 Feb 2025 11:12:40 +0900',
  'tag': ''},
 {'title': '전참시 연우 패딩 잠바 가방 현실도피자 후드티 옷 필라테스복 소파 쇼파 tv',
  'url': 'https://blog.naver.com/agla1/223747458810',
  'date': 'Tue, 04 Feb 2025 15:58:14 +0900',
  'tag': ''},
 {'title': '냉장고를 부탁해 손석구 토마토즙 레몬즙 시계 셔츠 옷 냉부해',
  'url': 'https://blog.naver.com/agla1/223746586407',
  'date': 'Mon, 03 Feb 2025 21:18:04 +0900',
  'tag': ''},
 {'title': '나혼자산다 김대호 새조개 샤브샤브 육수 여수 제철 유자 간장 소스 매생이 과메기 샤워 가운 염색약 나혼산',
  'url': 'https://blog.naver.com/agla1/223746379813',
  'date': 'Mon, 03 Feb 2025 17:36:08 +0900',
  'tag': ''},
 {'title': '미우새 이상민 한강 라면 기계 끓이는 조리기 이동건 엄마 패딩 고아라 셔츠 옷',
  'url': 'https://blog.naver.com/agla1/223745954402',
  'date': 'Mon, 03 Feb 2025 11:

In [11]:
import pandas as pd
l = []
for x, y in blogs:
    l = l + get_rss(x, y)
df = pd.DataFrame(l)
df

,title,url,date,tag
0,전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형,https://blog.naver.com/bcf5qp11/223750032612,"Fri, 07 Feb 2025 07:00:00 +0900","전지적참견시점,김아영,홈앤하비,도자기화병,모던인테리어,홈데코,감성인테리어,꽃꽂이,세..."
1,전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품,https://blog.naver.com/bcf5qp11/223750015985,"Thu, 06 Feb 2025 19:00:00 +0900","전지적참견시점,김아영,자사호다도세트,전통다도,차도구,보이차,우롱차,홍차,다도용품,찻..."
2,나완비 한지민 바지 짐머만 워시드 실크 벨티드 와이드 레그 팬츠,https://blog.naver.com/bcf5qp11/223748491659,"Thu, 06 Feb 2025 07:00:00 +0900","나의완벽한비서,한지민패션,짐머만팬츠,실크팬츠,벨티드팬츠,와이드레그팬츠,세련된스타일,..."
3,나완비 한지민 커피캔디 코피코 사탕 블리스터 커피맛사탕,https://blog.naver.com/bcf5qp11/223748472288,"Wed, 05 Feb 2025 19:00:00 +0900","나의완벽한비서,한지민,코피코사탕,커피맛사탕,블리스터팩,커피캔디,휴대용간식,무설"
4,전참시 김아영 마스크팩 웰라쥬 리얼 히알루로닉 블루 앰플 마스크,https://blog.naver.com/bcf5qp11/223747241692,"Wed, 05 Feb 2025 07:00:00 +0900","전지적참견시점,김아영마스크팩,웰라쥬,히알루론산마스크팩,보습케어,수분팩,피부탄력,건조..."
...,...,...,...,...
614,오윤아 TV 인생 향수 추천 여름 향수 엑스니힐로 메모 밀러 해리스 커정,https://blog.naver.com/unknown0998/223513096625,"Mon, 15 Jul 2024 10:24:48 +0900","오윤아여름향수,오윤아향수,엑스니힐로,엑스니힐로향수,밀러해리스향수,향수추천템,여름향수..."
615,이해리 부츠 패딩 조선양봉 꿀 거울 도마 (유튜브 속 아이템),https://blog.naver.com/nemo-c/223746366675,"Mon, 03 Feb 2025 17:24:28 +0900","거울,꿀선물,꿀선물세트,꿀,다이슨,다이슨드라이기,딥티크,화장대거울,다이슨드라이어,밤..."
616,전참시 연우 소파 쇼파 테이블 이불 피아노 스탠바이미 집 정보,https://blog.naver.com/nemo-c/223744671354,"Sat, 01 Feb 2025 23:48:01 +0900","이불,쇼파추천,쇼파,소파,패브릭소파,TV,스마트티비,디지털피아노,3인용쇼파,3인소파..."
617,나혼자산다 김대호 사우나 가격은? 가안 홈사우나 (구입링크),https://blog.naver.com/nemo-c/223743708954,"Fri, 31 Jan 2025 22:49:35 +0900","핀란드사우나,사우나스토브,건식사우나,사우나,가정용사우나,찜질방시공,핀란드식사우나,편..."


In [12]:
df['date'] = pd.to_datetime(df['date'])
df

,title,url,date,tag
0,전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형,https://blog.naver.com/bcf5qp11/223750032612,2025-02-07 07:00:00+09:00,"전지적참견시점,김아영,홈앤하비,도자기화병,모던인테리어,홈데코,감성인테리어,꽃꽂이,세..."
1,전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품,https://blog.naver.com/bcf5qp11/223750015985,2025-02-06 19:00:00+09:00,"전지적참견시점,김아영,자사호다도세트,전통다도,차도구,보이차,우롱차,홍차,다도용품,찻..."
2,나완비 한지민 바지 짐머만 워시드 실크 벨티드 와이드 레그 팬츠,https://blog.naver.com/bcf5qp11/223748491659,2025-02-06 07:00:00+09:00,"나의완벽한비서,한지민패션,짐머만팬츠,실크팬츠,벨티드팬츠,와이드레그팬츠,세련된스타일,..."
3,나완비 한지민 커피캔디 코피코 사탕 블리스터 커피맛사탕,https://blog.naver.com/bcf5qp11/223748472288,2025-02-05 19:00:00+09:00,"나의완벽한비서,한지민,코피코사탕,커피맛사탕,블리스터팩,커피캔디,휴대용간식,무설"
4,전참시 김아영 마스크팩 웰라쥬 리얼 히알루로닉 블루 앰플 마스크,https://blog.naver.com/bcf5qp11/223747241692,2025-02-05 07:00:00+09:00,"전지적참견시점,김아영마스크팩,웰라쥬,히알루론산마스크팩,보습케어,수분팩,피부탄력,건조..."
...,...,...,...,...
614,오윤아 TV 인생 향수 추천 여름 향수 엑스니힐로 메모 밀러 해리스 커정,https://blog.naver.com/unknown0998/223513096625,2024-07-15 10:24:48+09:00,"오윤아여름향수,오윤아향수,엑스니힐로,엑스니힐로향수,밀러해리스향수,향수추천템,여름향수..."
615,이해리 부츠 패딩 조선양봉 꿀 거울 도마 (유튜브 속 아이템),https://blog.naver.com/nemo-c/223746366675,2025-02-03 17:24:28+09:00,"거울,꿀선물,꿀선물세트,꿀,다이슨,다이슨드라이기,딥티크,화장대거울,다이슨드라이어,밤..."
616,전참시 연우 소파 쇼파 테이블 이불 피아노 스탠바이미 집 정보,https://blog.naver.com/nemo-c/223744671354,2025-02-01 23:48:01+09:00,"이불,쇼파추천,쇼파,소파,패브릭소파,TV,스마트티비,디지털피아노,3인용쇼파,3인소파..."
617,나혼자산다 김대호 사우나 가격은? 가안 홈사우나 (구입링크),https://blog.naver.com/nemo-c/223743708954,2025-01-31 22:49:35+09:00,"핀란드사우나,사우나스토브,건식사우나,사우나,가정용사우나,찜질방시공,핀란드식사우나,편..."


In [13]:
# date 2 days ago from now (date : dtype=datetime64[ns, UTC+09:00] usin np.datetime64
date_2d = pd.to_datetime('now', utc=True)
date_2d_ago = date_2d - pd.DateOffset(days=2)
date_2d_ago = date_2d_ago.floor('D')
df2 = df[df['date'] >= date_2d_ago]
df2.reset_index(drop=True, inplace=True)
df2.to_csv('blog_data/test2.csv')
df2

,title,url,date,tag
0,전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형,https://blog.naver.com/bcf5qp11/223750032612,2025-02-07 07:00:00+09:00,"전지적참견시점,김아영,홈앤하비,도자기화병,모던인테리어,홈데코,감성인테리어,꽃꽂이,세..."
1,전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품,https://blog.naver.com/bcf5qp11/223750015985,2025-02-06 19:00:00+09:00,"전지적참견시점,김아영,자사호다도세트,전통다도,차도구,보이차,우롱차,홍차,다도용품,찻..."
2,나완비 한지민 바지 짐머만 워시드 실크 벨티드 와이드 레그 팬츠,https://blog.naver.com/bcf5qp11/223748491659,2025-02-06 07:00:00+09:00,"나의완벽한비서,한지민패션,짐머만팬츠,실크팬츠,벨티드팬츠,와이드레그팬츠,세련된스타일,..."
3,나완비 한지민 커피캔디 코피코 사탕 블리스터 커피맛사탕,https://blog.naver.com/bcf5qp11/223748472288,2025-02-05 19:00:00+09:00,"나의완벽한비서,한지민,코피코사탕,커피맛사탕,블리스터팩,커피캔디,휴대용간식,무설"
4,강주은 시슬리 화장품 나스 컨실러 쉐도우 팔레트 헉슬리 크림 세럼 거울,https://blog.naver.com/celubdigging/223751391830,2025-02-07 14:53:45+09:00,"강주은,메이크업템,뷰티추천,강주은화장품,50대뷰티,자연스러운메이크업,럭셔리스킨케어,..."
5,강주은 드라이기 돈모 롤빗 브러쉬,https://blog.naver.com/celubdigging/223750065876,2025-02-06 18:06:45+09:00,"강주은,강주은곱슬머리,강주은드라이기,강주은헤어드라이어,강주은롤빗,강주은돈모브러쉬,강..."
6,나솔사계 경리 반팔 니트 미니스커트 정보,https://blog.naver.com/celubdigging/223749999680,2025-02-06 17:09:33+09:00,"나솔사계,나솔사계99화,경리패션,경리아가일니트,경리스커트,그로브,아가일패턴,봄코디,..."
7,이주미 애장템 핸드폰 케이스 샤워가운 자주 파자마 도시락통 멀티스틱 곱창 4색볼펜 ...,https://blog.naver.com/celubdigging/223749797103,2025-02-06 14:21:07+09:00,"이주미,이주미애장템,이주미애장품,이주미핸드폰케이스,이주미신지모루,이주미안경,이주미케..."
8,라디오스타 장도연 오프숄더 니트 코쿤 가디건 박나래 땡땡이 오프숄더 문세윤 꽃무늬 ...,https://blog.naver.com/celubdigging/223748800581,2025-02-05 17:42:57+09:00,"라디오스타장도연니트,라디오스타박나래블라우스,라디오스타코쿤가디건,라디오스타문세윤자켓,..."
9,애라원 두유제조기 신애라 찜기 혈당영양제 병아리콩 혈당측정기,https://blog.naver.com/celubdigging/223748586408,2025-02-05 14:49:22+09:00,"신애라찜기,신애라혈당영양제,신애라두유제조기,애라원혈당측정기,애라원건강템"


In [14]:
# date 1 days ago from now (date : dtype=datetime64[ns, UTC+09:00] usin np.datetime64
date_1d = pd.to_datetime('now', utc=True)
date_1d_ago = date_1d - pd.DateOffset(days=1)
date_1d_ago = date_1d_ago.floor('D')
df2 = df[df['date'] >= date_1d_ago]
df2.reset_index(drop=True, inplace=True)
df2.to_csv('blog_data/test2.csv')
df2

,title,url,date,tag
0,전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형,https://blog.naver.com/bcf5qp11/223750032612,2025-02-07 07:00:00+09:00,"전지적참견시점,김아영,홈앤하비,도자기화병,모던인테리어,홈데코,감성인테리어,꽃꽂이,세..."
1,전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품,https://blog.naver.com/bcf5qp11/223750015985,2025-02-06 19:00:00+09:00,"전지적참견시점,김아영,자사호다도세트,전통다도,차도구,보이차,우롱차,홍차,다도용품,찻..."
2,강주은 시슬리 화장품 나스 컨실러 쉐도우 팔레트 헉슬리 크림 세럼 거울,https://blog.naver.com/celubdigging/223751391830,2025-02-07 14:53:45+09:00,"강주은,메이크업템,뷰티추천,강주은화장품,50대뷰티,자연스러운메이크업,럭셔리스킨케어,..."
3,강주은 드라이기 돈모 롤빗 브러쉬,https://blog.naver.com/celubdigging/223750065876,2025-02-06 18:06:45+09:00,"강주은,강주은곱슬머리,강주은드라이기,강주은헤어드라이어,강주은롤빗,강주은돈모브러쉬,강..."
4,나솔사계 경리 반팔 니트 미니스커트 정보,https://blog.naver.com/celubdigging/223749999680,2025-02-06 17:09:33+09:00,"나솔사계,나솔사계99화,경리패션,경리아가일니트,경리스커트,그로브,아가일패턴,봄코디,..."
5,이주미 애장템 핸드폰 케이스 샤워가운 자주 파자마 도시락통 멀티스틱 곱창 4색볼펜 ...,https://blog.naver.com/celubdigging/223749797103,2025-02-06 14:21:07+09:00,"이주미,이주미애장템,이주미애장품,이주미핸드폰케이스,이주미신지모루,이주미안경,이주미케..."
6,시슬리 화장품만 쓰는 강주은이 추천하는 헉슬리,https://blog.naver.com/dehi61/223751438018,2025-02-07 15:22:02+09:00,"강주은,메이크업템,뷰티추천,강주은화장품,50대뷰티,자연스러운메이크업,럭셔리스킨케어,..."
7,강주은 charis 드라이기 2만원대 밖에 안하네요,https://blog.naver.com/dehi61/223750068994,2025-02-06 18:09:52+09:00,"강주은,강주은드라이기,가성비드라이기,강주은롤빗,돈모브러쉬,헤어관리,스타일링팁,볼륨드라이"
8,나솔사계 99화 경리 패션 찾았어요,https://blog.naver.com/dehi61/223750003834,2025-02-06 17:12:59+09:00,"나솔사계,나솔사계99화,경리패션,경리아가일니트,경리스커트,그로브,아가일패턴,봄코디,..."
9,이주미 자주 파자마 물티슈 파우치는 더 저렴한거 추천드림,https://blog.naver.com/dehi61/223749820907,2025-02-06 14:40:52+09:00,"이주미애장템,이주미추천템,이주미돈모브러쉬,이주미신지모루,이주미도시락,이주미파자마,이..."


In [ ]:
from openai import OpenAI

api_key = ""
client = OpenAI(api_key=api_key)


def extract_celeb(title):
    print(title)
    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": "문구를 입력하면 그 문구로부터 연예인의 이름을 찾아서 출력해줘. 문장이 나와도 문장에 대답하지 말고, 해당 문장에 등장하는 연예인의 이름만 찾아서 출력해.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "한지민 올리브데올리브 언발란스 니트 셋업 풀오버 나완비 코디",
                    }
                ],
            },
            {"role": "assistant", "content": [{"type": "text", "text": "한지민"}]},
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "독수리5형제를부탁해 1회 엄지원 옷 니트 스웨터 가디건 폴로 숄더백 가방 시계 마광숙 패션 코디 의상",
                    }
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": "엄지원, 마광숙"}],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "김대호 나혼자산다 하차 선물 가격은?",
                    }
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": "김대호"}],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": title,
                    }
                ],
            },
        ],
        response_format={"type": "text"},
        temperature=1,
        max_completion_tokens=10000,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )
    ans = response.choices[0].message.content.strip()
    print(ans)
    return ans

In [16]:
resp = extract_celeb('나혼자산다 김대호 집 사우나 기계 건습식 맥반석 건식 습식 핀란드 사우나기 나혼산 대호 아나운서 찜질방')
resp

나혼자산다 김대호 집 사우나 기계 건습식 맥반석 건식 습식 핀란드 사우나기 나혼산 대호 아나운서 찜질방
김대호


'김대호'

In [17]:
df2['celeb'] = df2['title'].apply(extract_celeb)
df2.to_csv('blog_data/test3.csv')
df2

전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형
김아영
전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품
김아영
강주은 시슬리 화장품 나스 컨실러 쉐도우 팔레트 헉슬리 크림 세럼 거울


KeyboardInterrupt: 

In [1]:
import pandas as pd
df2 = pd.read_csv('blog_data/test3.csv', index_col=0)
df2

,title,url,date,tag,celeb
0,전참시 김아영 화병 홈앤하비 도자기화병 세라믹 인테리어 고리형,https://blog.naver.com/bcf5qp11/223750032612,2025-02-07 07:00:00+09:00,"전지적참견시점,김아영,홈앤하비,도자기화병,모던인테리어,홈데코,감성인테리어,꽃꽂이,세...",김아영
1,전참시 김아영 다도세트 자사호 전통다기세트 찻잔세트 다도용품,https://blog.naver.com/bcf5qp11/223750015985,2025-02-06 19:00:00+09:00,"전지적참견시점,김아영,자사호다도세트,전통다도,차도구,보이차,우롱차,홍차,다도용품,찻...",김아영
2,강주은 시슬리 화장품 나스 컨실러 쉐도우 팔레트 헉슬리 크림 세럼 거울,https://blog.naver.com/celubdigging/223751391830,2025-02-07 14:53:45+09:00,"강주은,메이크업템,뷰티추천,강주은화장품,50대뷰티,자연스러운메이크업,럭셔리스킨케어,...",강주은
3,강주은 드라이기 돈모 롤빗 브러쉬,https://blog.naver.com/celubdigging/223750065876,2025-02-06 18:06:45+09:00,"강주은,강주은곱슬머리,강주은드라이기,강주은헤어드라이어,강주은롤빗,강주은돈모브러쉬,강...",강주은
4,나솔사계 경리 반팔 니트 미니스커트 정보,https://blog.naver.com/celubdigging/223749999680,2025-02-06 17:09:33+09:00,"나솔사계,나솔사계99화,경리패션,경리아가일니트,경리스커트,그로브,아가일패턴,봄코디,...",경리
5,이주미 애장템 핸드폰 케이스 샤워가운 자주 파자마 도시락통 멀티스틱 곱창 4색볼펜 ...,https://blog.naver.com/celubdigging/223749797103,2025-02-06 14:21:07+09:00,"이주미,이주미애장템,이주미애장품,이주미핸드폰케이스,이주미신지모루,이주미안경,이주미케...",이주미
6,시슬리 화장품만 쓰는 강주은이 추천하는 헉슬리,https://blog.naver.com/dehi61/223751438018,2025-02-07 15:22:02+09:00,"강주은,메이크업템,뷰티추천,강주은화장품,50대뷰티,자연스러운메이크업,럭셔리스킨케어,...",강주은
7,강주은 charis 드라이기 2만원대 밖에 안하네요,https://blog.naver.com/dehi61/223750068994,2025-02-06 18:09:52+09:00,"강주은,강주은드라이기,가성비드라이기,강주은롤빗,돈모브러쉬,헤어관리,스타일링팁,볼륨드라이",강주은
8,나솔사계 99화 경리 패션 찾았어요,https://blog.naver.com/dehi61/223750003834,2025-02-06 17:12:59+09:00,"나솔사계,나솔사계99화,경리패션,경리아가일니트,경리스커트,그로브,아가일패턴,봄코디,...",경리
9,이주미 자주 파자마 물티슈 파우치는 더 저렴한거 추천드림,https://blog.naver.com/dehi61/223749820907,2025-02-06 14:40:52+09:00,"이주미애장템,이주미추천템,이주미돈모브러쉬,이주미신지모루,이주미도시락,이주미파자마,이...",이주미


In [3]:
celebs = []
for i in df2['celeb'].to_list():
    celebs += i.split(', ')
k = pd.DataFrame(celebs).value_counts()
k

0   
최화정     9
강주은     6
경리      4
이주미     3
송해나     2
김아영     2
장도연     2
ITZY    1
데프콘     1
신애라     1
영숙      1
아이브     1
아이키     1
양세찬     1
유나      1
옥순      1
영자      1
유세윤     1
이현이     1
이이경     1
정숙      1
한혜진     1
Name: count, dtype: int64

In [4]:
tk = [x[0] for x in k.keys()][:3]
tk

['최화정', '강주은', '경리']

In [5]:
keyword = tk[0]

In [6]:
dft = df2[df2['title'].str.contains(keyword)].reset_index(drop=True)
dft

,title,url,date,tag,celeb
0,최화정 석쇠 투명지갑 향수 진주 목걸이 귀걸이 블러셔 아로마오일 가방 괄사 핸드크림...,https://blog.naver.com/hkh443/223750217199,2025-02-06 20:59:10+09:00,"최화정,최화정인생템,최화정석쇠,최화정투명지갑,최화정향수,최화정진주목걸이,최화정진주귀...",최화정
1,최화정 절구통 구찌 장갑 수동 착즙기 오일가글 유튜브 안녕하세요,https://blog.naver.com/hprbel1097/223750429455,2025-02-06 23:50:58+09:00,"최화정오일가글,최화정가글,최화정동국제약,최화정절구통,최화정착즙기,최화정수동착즙기,최...",최화정
2,최화정 퍼퓸 향수 셀린느 핸드백 버킷백 가방 보석 파우치 머리핀 안경 선글라스 유튜...,https://blog.naver.com/hprbel1097/223750407473,2025-02-06 23:33:04+09:00,"최화정보석함,최화정보석파우치,최화정퍼퓸,최화정향수,최화정셀린느,최화정핸드백,최화정가...",최화정
3,최화정 마사지기 루즈 보관용 투명지갑 목걸이 미니석쇠 앞치마 블러쉬 핸드크림 아로마...,https://blog.naver.com/hprbel1097/223750377422,2025-02-06 23:10:10+09:00,"최화정블러쉬,최화정목걸이,최화정반클리프,최화정설화수,최화정마사지기,최화정앞치마,최화...",최화정
4,최화정 향수 퍼퓸 킬리안 바이레도 괄사 마사지기 설화수 진옥 백옥 마사지 아로마 오일,https://blog.naver.com/hsh6566/223751314968,2025-02-07 14:05:52+09:00,"최화정향수,최화정괄사,최화정아로마오일",최화정
5,최화정 화로 가정용 미니 석쇠 구이 그릴 레몬 착즙기 레몬즙 짜개 스퀴저 즙짜개 접...,https://blog.naver.com/hsh6566/223750984544,2025-02-07 10:25:27+09:00,"최화정화로,최화정석쇠,최화정그릴,최화정레몬착즙기,최화정레몬즙짜개",최화정
6,최화정 투명 지갑 비닐 파우치 동전 자라 버킷백 셀린느 가방 구찌 장갑 블러셔 앞머...,https://blog.naver.com/hsh6566/223750871127,2025-02-07 08:52:55+09:00,"최화정투명지갑,최화정버킷백,최화정구찌장갑,최화정블러셔,최화정찍찍이",최화정
7,최화정 진주 목걸이 반클리프 터키석 터콰이즈 귀걸이 뿔테 안경 카린 선글라스 인생템,https://blog.naver.com/hsh6566/223750315721,2025-02-07 06:30:00+09:00,"최화정진주목걸이,최화정반클리프목걸이,최화정진주귀걸이,최화정안경,최화정선글라스",최화정
8,최화정 미니석쇠 그릴 투명지갑 괄사 목걸이 귀걸이 가방 버킷백 핸드크림 아로마오일 ...,https://blog.naver.com/lzheng/223750201349,2025-02-06 20:40:40+09:00,"9101,최화정미니석쇠그릴,최화정투명지갑,최화정괄사,최화정목걸이,최화정귀걸이,최화정...",최화정


In [7]:
keywords = []
for i in dft.title:
    for j in i.split(' '):
        keywords.append(j)
k = pd.DataFrame(keywords).value_counts()
k

0  
최화정    9
목걸이    4
향수     4
가방     4
괄사     3
      ..
킬리안    1
카린     1
하트     1
핸드백    1
화로     1
Name: count, Length: 73, dtype: int64

In [45]:
import time, random
from fake_useragent import UserAgent

def crawling_waiting_time() -> None:
    return time.sleep(random.randint(1, 3))

def get_html(url):
    import requests

    # pretend headers
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": "https://blog.naver.com/",
        "Connection": "keep-alive",
        "Cache-Control": "max-age=0",
        "Upgrade-Insecure-Requests": "1",
    }
    response = requests.get(url, headers=headers, timeout=10)
    rurl = 'https://blog.naver.com/' + response.text.split('src="')[2].split('"\n')[0]
    print(rurl)
    response = requests.get(rurl, headers=headers, timeout=10)
    return response.text



In [14]:
def get_content(html):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html.text, 'html.parser')
    output = '\n'.join([x.get_text() for x in soup.find_all("p", {"class":"se-text-paragraph"})])
    return output.replace('\u200b', '')

In [57]:
html = get_html('https://blog.naver.com/hkh443/223750217199	')
html

https://blog.naver.com//PostView.naver?blogId=hkh443&logNo=223750217199&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false


'\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\r\n<html lang="ko">\r\n<head>\r\n<meta http-equiv="Pragma" content="no-cache" />\r\n<meta http-equiv="Expires" content="-1" />\r\n\r\n<meta name="referrer" content="always" />\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n<!--[if ie]>\r\n<style type="text/css">\r\nhtml {overflow: scroll; overflow-x: auto;}\r\n</style>\r\n<![endif]-->\r\n\r\n<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/t.static.blog/mylog/versioning/LayoutTopCommon-233548907_https.css" charset="UTF-8" />\r\n<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/t.static.blog/mylog/versioning//common/css/music/player-d3fc09e_https.css">\r\n\r\n<link rel="shortcut icon" type="image/x-icon" href="/favicon.ico?3" />\r\n\n\n\n\n\n \n\n\t\n\t\t<meta property="og:title" content="최화정 석쇠 투명지갑 향수 진주 목걸이 귀걸이 블러셔 아로마오일 가방 괄사 핸드크림 하트 앞머리 찍찍이 선글라스 인생템"/>\n\n\t\t\n\t\t\t\n\t\t\t\n\t\t\t

In [53]:
def get_redirect(url):
    crawling_waiting_time()
    import requests

    # pretend headers with private mode
    
    # headers = {
    #     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    #     "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    #     "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    #     "Referer": "https://blog.naver.com/",
    #     "Connection": "keep-alive",
    #     "Cache-Control": "max-age=0",
    #     "Upgrade-Insecure-Requests": "1",
    # }
    try:
        location = ''
        user = UserAgent().random
        headers = {
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
            "Accept-Language": "ko-KR,ko;q=0.8,en-US;q=0.5,en;q=0.3"
        }
        r = requests.get(url, headers=headers, allow_redirects=False, timeout=10)
        print(r.headers)
        if 'location' in r.headers:
            location = r.headers['location']
        elif 'Location' in r.headers:
            location = r.headers['Location']
        return location
    except:
        return ''
    # r = requests.get(url, headers=headers, allow_redirects=False, timeout=10)
    # return r.url

In [54]:
def get_item_name(url):
    import urllib.parse
    try:
        return urllib.parse.unquote_plus(url.split('q=')[1].split('&src=')[0])
    except:
        try:
            return urllib.parse.unquote_plus(url.split('&pageKey=')[1].split('&')[0])
        except:
            return ''
    

In [62]:
def get_href(html):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, 'html.parser')
    elems = soup.find_all("a", {"class":"se-link"})
    print(len(elems))
    # output = [get_redirect(x.get('href')) for x in elems]
    # print(output)
    output_dict = {}
    for x in elems:
        if x != '':
            href = x.get('href')
            print(f'href : {href}')
            redir = get_redirect(href)
            print(f' - redir : {redir}')
            if 'link.coupang' in redir:
                redir = get_redirect(redir)
                print(f' - (re)redir : {redir}')
            name = get_item_name(redir).replace(u'\xa0', u' ')
            output_dict[x.get_text()] = name

    output_dict = {x for x in output_dict.items() if x[1] != ''}
    return output_dict

In [63]:
links = get_href(html)
links

18
href : https://vvd.bz/jEV
{'Date': 'Fri, 07 Feb 2025 08:32:51 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Vary': 'Origin, Access-Control-Request-Method, Access-Control-Request-Headers', 'Set-Cookie': 'linkAgentKey=653a85a2ed9b46c1b7b7bc8dcd202829; Max-Age=86400; Expires=Sat, 08 Feb 2025 08:32:51 GMT; Path=/; Secure; HttpOnly', 'Location': 'https://link.coupang.com/a/cdnhva', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Content-Language': 'ko', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': 'frame-ancestors *', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'Accept-CH': 'Sec-CH-UA-Model', 'Permissions-Policy': 'ch-ua-model=(self "https://vvd.bz")', 'Referrer-Policy': 'unsafe-url', 'Alt-Svc': 'h3=":443"; ma=86400'}
 - redir : https://link.coupang.com/a/cdnhva
{'Content-Length': '0', 'Server': 'istio-envoy', 'x-ua-compatible': 'IE=edge', 'Cache-Control': 'no-store, no-cache, must-revalidate', 'x-robots-tag': 'no

{('\u200b', '킬리안 러브 돈 비 샤이 향수'),
 ('▶괄사 마사저 보러가기', '미니 괄사'),
 ('▶귀걸이 보러가기', '왕진주귀걸이'),
 ('▶목걸이 보러가기', '터키석 목걸이'),
 ('▶선글라스 보러가기', 'CARIN KRISTEN  선글라스'),
 ('▶최화정 레몬 즙짜개 보러가기', '드림팜 플루서'),
 ('▶최화정 블러셔 보러가기', '조르지오 아르마니 a블러쉬'),
 ('▶최화정 석쇠 보러가기', '마루쥬 카나아미 원적외선 세라믹 석쇠'),
 ('▶최화정 아로마 오일 보러가기', '피몽쉐 리버티'),
 ('▶최화정 투명지갑 보러가기', '투명 동전지갑 파우치'),
 ('▶최화정 하트 찍찍이 보러가기', '하트 모양 벨크로 찍찍이'),
 ('▶최화정 핸드크림 보러가기', '블리쉐던 핸드크림'),
 ('▶최화정 향수 보러가기', '바이레도 블랑쉬 EDP 50ml')}

In [41]:
d3 = df3.to_dict()
d3[0].content

b'\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\r\n<html lang="ko">\r\n<head>\r\n<meta http-equiv="Pragma" content="no-cache" />\r\n<meta http-equiv="Expires" content="-1" />\r\n\r\n<meta name="referrer" content="always" />\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n<!--[if ie]>\r\n<style type="text/css">\r\nhtml {overflow: scroll; overflow-x: auto;}\r\n</style>\r\n<![endif]-->\r\n\r\n<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/t.static.blog/mylog/versioning/LayoutTopCommon-233548907_https.css" charset="UTF-8" />\r\n<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/t.static.blog/mylog/versioning//common/css/music/player-d3fc09e_https.css">\r\n\r\n<link rel="shortcut icon" type="image/x-icon" href="/favicon.ico?3" />\r\n\n\n\n\n\n \n\n\t\n\t\t<meta property="og:title" content="\xec\xa0\x84\xec\xb0\xb8\xec\x8b\x9c \xea\xb9\x80\xec\x95\x84\xec\x98\x81 \xed\x99\x94\xeb\xb3\x91 \

In [44]:
df3 = dft['url'].apply(get_html)
df3.to_csv('blog_data/htmllist.csv')
df3

https://blog.naver.com//PostView.naver?blogId=hkh443&logNo=223750217199&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false
https://blog.naver.com//PostView.naver?blogId=hprbel1097&logNo=223750429455&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false
https://blog.naver.com//PostView.naver?blogId=hprbel1097&logNo=223750407473&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false
https://blog.naver.com//PostView.naver?blogId=hprbel1097&logNo=223750377422&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false
https://blog.naver.com//PostView.naver?blogId=hsh6566&logNo=223751314968&redirect=Dlog&widgetTypeCall=true&topReferer=https%3A%2F%2Fblog.naver.com%2F&trackingCode=blog_etc&directAccess=false
https://blog.naver.com//PostView.nave

0    \r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//...
1    \r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//...
2    \r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//...
3    \r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC "-//...
4    \r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC ...
5    \r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC ...
6    \r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC ...
7    \r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC ...
8    \r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE HTML PUBLIC ...
Name: url, dtype: object

In [51]:
df4 = df3.apply(get_href)
df4.to_csv('blog_data/itemlist.csv')
df4

18
href : https://vvd.bz/jEV
{'Date': 'Fri, 07 Feb 2025 08:24:51 GMT', 'Connection': 'keep-alive', 'Vary': 'Origin, Access-Control-Request-Method, Access-Control-Request-Headers', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': 'frame-ancestors *', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'Accept-CH': 'Sec-CH-UA-Model', 'Permissions-Policy': 'ch-ua-model=(self "https://vvd.bz")', 'Referrer-Policy': 'unsafe-url', 'Alt-Svc': 'h3=":443"; ma=86400'}
 - redir : 
href : https://vvd.bz/jEY
{'Date': 'Fri, 07 Feb 2025 08:24:54 GMT', 'Connection': 'keep-alive', 'Vary': 'Origin, Access-Control-Request-Method, Access-Control-Request-Headers', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': 'frame-ancestors *', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'Accept-CH': 'Sec-CH-UA-Model',

KeyboardInterrupt: 